# 🟢 Day 2 — State & Reducers

**LangGraph from Zero to Advanced · Day 2 of 16**

**Today's goal:** control *how* the state changes when nodes write to it.

| Lesson | Topic |
|---|---|
| 1 | Default rule: overwrite |
| 2 | Reducers: append instead of overwrite |
| 3 | Your own reducer |
| 4 | Why reducers matter: nodes running in parallel |
| 5 | Default values & optional keys |
| 6 | Input and output schemas (hide internal keys) |
| 7 | Mini project: shopping cart |
| 8 | Practice |

No API key needed today.

In [ ]:
%pip install -q -U langgraph

In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## Lesson 1 — Default rule: overwrite

If two nodes write to the same key, the **last one wins**. This is fine for single values like a score — but bad for a list you want to grow.

In [ ]:
class LogState(TypedDict):
    log: list[str]

def step_a(state):
    return {"log": ["A ran"]}

def step_b(state):
    return {"log": ["B ran"]}

def build(state_class):
    g = StateGraph(state_class)
    g.add_sequence([step_a, step_b])
    g.add_edge(START, "step_a")
    g.add_edge("step_b", END)
    return g.compile()

print(build(LogState).invoke({"log": []}))   # "A ran" is lost!

## Lesson 2 — Reducers: append instead of overwrite

A **reducer** is a function `(old_value, new_value) -> merged_value`. You attach it with `Annotated[type, reducer]`.

`operator.add` on two lists = join them. So the list **grows** instead of being replaced.

In [ ]:
class AppendLogState(TypedDict):
    log: Annotated[list[str], operator.add]   # 👈 the only change

result = build(AppendLogState).invoke({"log": []})
print(result)
assert result["log"] == ["A ran", "B ran"]
print("✅ Reducer appends!")

| State key | Rule | Result |
|---|---|---|
| `log: list[str]` | overwrite | `['B ran']` |
| `log: Annotated[list[str], operator.add]` | append | `['A ran', 'B ran']` |

`operator.add` also works for numbers — it **adds** them. Great for counters:

In [ ]:
class CounterState(TypedDict):
    total: Annotated[int, operator.add]

def plus_five(state):
    return {"total": 5}

def plus_ten(state):
    return {"total": 10}

g = StateGraph(CounterState)
g.add_sequence([plus_five, plus_ten])
g.add_edge(START, "plus_five")
g.add_edge("plus_ten", END)
print(g.compile().invoke({"total": 100}))   # 100 + 5 + 10

## Lesson 3 — Your own reducer

Any function with two inputs works. Example: keep only the **highest** exam score.

In [ ]:
def keep_max(old: int, new: int) -> int:
    return max(old, new)

class ExamState(TypedDict):
    best_score: Annotated[int, keep_max]

def exam_1(state): return {"best_score": 72}
def exam_2(state): return {"best_score": 65}   # lower — should be ignored
def exam_3(state): return {"best_score": 88}

g = StateGraph(ExamState)
g.add_sequence([exam_1, exam_2, exam_3])
g.add_edge(START, "exam_1")
g.add_edge("exam_3", END)
res = g.compile().invoke({"best_score": 0})
print(res)
assert res["best_score"] == 88
print("✅ Custom reducer works!")

## Lesson 4 — Why reducers matter: parallel nodes

If one node has edges to **two** nodes, both run **at the same time** (in the same "step"). This is called **fan-out**. When both finish, the graph continues (**fan-in**).

```
          ┌──► check_weather ──┐
START ────┤                    ├──► summary ──► END
          └──► check_traffic ──┘
```

If both parallel nodes write the same key **without** a reducer, LangGraph can't decide who wins → error. With a reducer, both updates are merged.

In [ ]:
from langgraph.errors import InvalidUpdateError

def make_trip_graph(state_class):
    def check_weather(state): return {"notes": ["☀️ sunny"]}
    def check_traffic(state): return {"notes": ["🚗 light traffic"]}
    def summary(state):       return {"notes": [f"{len(state['notes'])} checks done"]}

    g = StateGraph(state_class)
    g.add_node("check_weather", check_weather)
    g.add_node("check_traffic", check_traffic)
    g.add_node("summary", summary)
    g.add_edge(START, "check_weather")     # two edges from START = fan-out
    g.add_edge(START, "check_traffic")
    g.add_edge("check_weather", "summary") # both lead to summary = fan-in
    g.add_edge("check_traffic", "summary")
    g.add_edge("summary", END)
    return g.compile()

class NoReducer(TypedDict):
    notes: list[str]

try:
    make_trip_graph(NoReducer).invoke({"notes": []})
except InvalidUpdateError as e:
    print("❌ Without reducer:", type(e).__name__)

In [ ]:
class WithReducer(TypedDict):
    notes: Annotated[list[str], operator.add]

trip = make_trip_graph(WithReducer)
show_graph(trip)
res = trip.invoke({"notes": []})
print(res)
assert len(res["notes"]) == 3
print("✅ Parallel nodes merged safely!")

💡 **Rule of thumb:** if a key can be written by more than one node in the same step, give it a reducer.

## Lesson 5 — Default values & optional keys

You don't have to pass every key in `invoke()`. But if a node reads a key that was never set, you get a `KeyError`. Use `state.get(key, default)` to be safe.

In [ ]:
class Visit(TypedDict):
    name: str
    visits: int

def count_visit(state: Visit) -> dict:
    return {"visits": state.get("visits", 0) + 1}   # safe default

g = StateGraph(Visit)
g.add_node("count_visit", count_visit)
g.add_edge(START, "count_visit")
g.add_edge("count_visit", END)
visit_app = g.compile()

print(visit_app.invoke({"name": "Meera"}))              # visits not given
print(visit_app.invoke({"name": "Meera", "visits": 4}))

💡 You can also use a **Pydantic model** as state. It validates types and supports default values:

In [ ]:
from pydantic import BaseModel

class Order(BaseModel):
    item: str
    qty: int = 1          # default value

def describe(state: Order) -> dict:
    return {"item": f"{state.qty} x {state.item}"}   # Pydantic: use state.qty, not state["qty"]

g = StateGraph(Order)
g.add_node("describe", describe)
g.add_edge(START, "describe")
g.add_edge("describe", END)
print(g.compile().invoke({"item": "masala dosa"}))

## Lesson 6 — Input and output schemas

Sometimes the graph needs internal "scratch" keys the user should not see. Give the graph separate **input** and **output** schemas:

In [ ]:
class InputState(TypedDict):
    question: str

class OutputState(TypedDict):
    answer: str

class FullState(InputState, OutputState):
    scratch_notes: str          # internal only

def think(state: FullState) -> dict:
    return {"scratch_notes": f"user asked: {state['question']}"}

def respond(state: FullState) -> dict:
    return {"answer": f"Here is my answer to '{state['question']}'"}

g = StateGraph(FullState, input_schema=InputState, output_schema=OutputState)
g.add_sequence([think, respond])
g.add_edge(START, "think")
g.add_edge("respond", END)
res = g.compile().invoke({"question": "What is LangGraph?"})
print(res)
assert "scratch_notes" not in res
print("✅ Internal keys are hidden from the output!")

## Lesson 7 — Mini project: shopping cart

Three "shelves" add items in parallel, then a checkout node calculates the bill.

In [ ]:
class Cart(TypedDict):
    items: Annotated[list[str], operator.add]
    total: Annotated[int, operator.add]
    bill: str

def fruits(state):     return {"items": ["apple"], "total": 120}
def dairy(state):      return {"items": ["milk", "paneer"], "total": 60 + 90}
def snacks(state):     return {"items": ["chips"], "total": 20}

def checkout(state):
    return {"bill": f"{len(state['items'])} items, ₹{state['total']}"}

g = StateGraph(Cart)
for fn in (fruits, dairy, snacks, checkout):
    g.add_node(fn.__name__, fn)
for shelf in ("fruits", "dairy", "snacks"):
    g.add_edge(START, shelf)
    g.add_edge(shelf, "checkout")
g.add_edge("checkout", END)
cart_app = g.compile()
show_graph(cart_app)

res = cart_app.invoke({"items": [], "total": 0})
print(res)
assert res["bill"] == "4 items, ₹290"
print("✅ Mini project done!")

## Lesson 8 — Practice

**Exercise 1.** Write a reducer `unique_add(old, new)` that appends only items not already in the list. Use it for `tags`, with two nodes that both add `"python"`.

**Exercise 2.** Build a graph with three parallel "judges" that each give a score (7, 9, 8). Store all scores in a list, then an `average` node computes the mean.

In [ ]:
# ✅ Solution 1
def unique_add(old: list, new: list) -> list:
    return old + [x for x in new if x not in old]

class TagState(TypedDict):
    tags: Annotated[list[str], unique_add]

def tag_a(state): return {"tags": ["python", "ai"]}
def tag_b(state): return {"tags": ["python", "langgraph"]}

g = StateGraph(TagState)
g.add_sequence([tag_a, tag_b])
g.add_edge(START, "tag_a")
g.add_edge("tag_b", END)
res = g.compile().invoke({"tags": []})
print(res)
assert res["tags"] == ["python", "ai", "langgraph"]

In [ ]:
# ✅ Solution 2
class JudgeState(TypedDict):
    scores: Annotated[list[int], operator.add]
    average: float

g = StateGraph(JudgeState)
for name, score in [("judge_1", 7), ("judge_2", 9), ("judge_3", 8)]:
    g.add_node(name, lambda state, s=score: {"scores": [s]})
    g.add_edge(START, name)
    g.add_edge(name, "average")
g.add_node("average", lambda state: {"average": sum(state["scores"]) / len(state["scores"])})
g.add_edge("average", END)
res = g.compile().invoke({"scores": []})
print(res)
assert res["average"] == 8.0
print("✅ All exercises solved!")

## 🎯 Day 2 Recap

* Default: a new value **overwrites** the old one.
* `Annotated[list, operator.add]` → **append**. Write your own reducer for any rule.
* Two edges from one node → nodes run **in parallel**; shared keys need a reducer.
* Use `state.get()` for optional keys; Pydantic for validation & defaults.
* `input_schema` / `output_schema` hide internal keys.

**Tomorrow (Day 3):** let the graph **make decisions** — branches and loops.